# DARF v3 — Face-Aware Two-Stage with Background Lock

v1 Test C (two-stage layout → identity) en iyi visual sonucu vermişti, ama:
- Stage 2'de Daenerys hâlâ küçük yan dönme yapıyordu
- 3. kişi arka planda hâlâ çıkıyordu

**v3 hipotezi:**
1. Stage 1'in pose'unu **face-aware skeleton** (eye-eye, ear-ear, nose-neck lineları + opsiyonel jaw/brow/mouth) ile vererek kafa düz bakacak şekilde sabitle
2. Stage 1'de **background latent lock** ile pose dışında 3. kişi inşasını engelle
3. Stage 2'de **per-block floor** (down=0.15, mid=0.08, up=0.0) → kompozisyon coherence + identity isolation
4. Stage 2 `strength=0.40`, `identity_ctrl_scale=0.95` → Stage 1'in pose'unu koru, sadece kimliği refine et

Bu notebook **C tabanlı** ama her aşamasına v2 mekanizmalarını entegre ediyor.

## Bölüm 1 — Setup

In [ ]:
import os
REPO_URL = "https://github.com/melik3kara/LoRA-scale-adaptive-feedback.git"
REPO_DIR = "/workspace/LoRA-scale-adaptive-feedback"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo at {REPO_DIR}, pulling...")
    !cd {REPO_DIR} && git pull

In [ ]:
!pip install -q gdown
os.makedirs(f"{REPO_DIR}/data/loras", exist_ok=True)
DRIVE_FOLDER_ID = "1L6NA_AT5HGSNOcKjw7gECbPKnaN86Ix5"
import glob
if not glob.glob(f"{REPO_DIR}/data/loras/*.safetensors"):
    !gdown --folder "https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}" -O {REPO_DIR}/data/loras/
    import shutil
    for f in glob.glob(f"{REPO_DIR}/data/loras/**/*.safetensors", recursive=True):
        target = os.path.join(f"{REPO_DIR}/data/loras", os.path.basename(f))
        if f != target:
            shutil.move(f, target)
!ls -lh {REPO_DIR}/data/loras/

In [ ]:
os.environ["HF_HOME"] = "/workspace/cache/huggingface"
os.environ["INSIGHTFACE_HOME"] = "/workspace/cache/insightface"
os.makedirs("/workspace/cache/huggingface", exist_ok=True)
os.makedirs("/workspace/cache/insightface", exist_ok=True)

!pip install -q diffusers==0.30.3 transformers==4.44.2 peft controlnet-aux insightface onnxruntime-gpu
!pip install -q "pydantic<2.10" "typing_extensions>=4.13.0"
!pip install -q "mediapipe==0.10.14" "protobuf<5" scipy

In [ ]:
os.chdir(REPO_DIR)
import sys
if "src" not in sys.path:
    sys.path.insert(0, "src")
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

## Bölüm 2 — Pipeline + Scorers + Evaluator

In [ ]:
from pipeline import load_identities, build_pipeline
from scorer import FaceScorer
from pose_scorer import PoseScorer
from attribute_scorer import AttributeScorer
from evaluator import Evaluator
from PIL import Image
from IPython.display import display

identities = load_identities()
pipe = build_pipeline(identities)

face_scorer = FaceScorer(reference_dir="data/reference_faces", device="cuda")
pose_scorer = PoseScorer()
attr_scorer = AttributeScorer(identities, device="cuda")
evaluator   = Evaluator(device="cuda")

ref_embeds = {k: evaluator.extract_face_embedding(
    Image.open(identities[k]["reference_image"]).convert("RGB")) for k in identities}

names = list(identities.keys())
identity_regions = {names[0]: (0,0,512,1024), names[1]: (512,0,1024,1024)}
print("\nReady.")

In [ ]:
import numpy as np, cv2
def detected_face_count(img):
    img_bgr = cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2BGR)
    return len(evaluator.face_app.get(img_bgr))

def show_evaluator_scores(label, image):
    sim_h, sim_d = evaluator.detect_and_assign_faces(
        image, ref_embeds["hermione"], ref_embeds["daenerys"]
    )
    n = detected_face_count(image)
    print(f"[Eval/{label}] arc_h={sim_h:+.3f}  arc_d={sim_d:+.3f}  faces_detected={n}")
    return {"label": label, "sim_h": sim_h, "sim_d": sim_d, "faces": n}

## Bölüm 3 — Face-Aware Pose'u Üret

İki versiyon — `dense_face=True` jaw, brows, mouth, nose bridge çizgilerini de ekler.

In [ ]:
from face_aware_pose import make_face_aware_pose, get_face_aware_target_keypoints

pose_simple = make_face_aware_pose(1024, 1024, front_facing=True, dense_face=False)
pose_dense  = make_face_aware_pose(1024, 1024, front_facing=True, dense_face=True)

os.makedirs("data/pose_images", exist_ok=True)
pose_simple.save("data/pose_images/face_aware_simple.png")
pose_dense.save("data/pose_images/face_aware_dense.png")

target_keypoints = get_face_aware_target_keypoints(front_facing=True)

print("Face-aware (eye-line + ear-line + nose-neck):")
display(pose_simple.resize((384, 384)))
print("Dense face (+ jawline, brows, mouth, nose bridge):")
display(pose_dense.resize((384, 384)))

## Bölüm 4 — Tek Konfigürasyon Test (asıl çalıştırma)

v3'ün bütün mekanizmaları açık, dense face pose, BG lock Stage 1'de aktif.

**Kontrol noktaları:**
- Stage 1 image: 2 jenerik kişi, 3. kişi yok (BG lock), kafalar düz
- Stage 2 image: identity belirgin, saç renkleri farklı, pose Stage 1'inkiyle aynı
- `faces_detected=2` ideal

In [ ]:
from pipeline import generate_two_stage

stage1, stage2 = generate_two_stage(
    pipe, identities, pose_dense,                    # ← face-aware DENSE pose
    layout_lora_scales={"hermione": 0.10, "daenerys": 0.30},
    identity_lora_scales={
        "hermione": {"down": 0.15, "mid": 0.30, "up": 0.45},
        "daenerys": {"down": 0.55, "mid": 1.20, "up": 1.50},
    },
    layout_ctrl_scale=1.00,
    identity_ctrl_scale=0.95,                        # pose lock sıkı
    refine_strength=0.40,                             # Stage 1 korunur
    refine_steps=28,
    layout_steps=28,
    seed=42,
    use_regional_attention=True,
    use_spatial_lora_gate=True,
    spatial_gate_block_floor={"down": 0.15, "mid": 0.08, "up": 0.00},
    use_bg_lock=True,
    bg_lock_ratio=0.40,                               # ilk %40 step'te BG dondur
    bg_lock_padding=24,
    bg_lock_in_layout=True,
    bg_lock_in_identity=False,
    identity_regions=identity_regions,
)

print("\n[Stage 1 — layout]");  display(stage1.resize((512, 512)))
print("[Stage 2 — identity]");  display(stage2.resize((512, 512)))
show_evaluator_scores("v3_main_stage1", stage1)
show_evaluator_scores("v3_main_stage2", stage2)

## Bölüm 5 — Pose Ablation

Aynı seed, aynı her şey, sadece pose değişiyor:

| Pose | Beklenti |
|---|---|
| `pose_simple` (eye/ear/nose lineları) | Daenerys daha az yan dönmeli |
| `pose_dense` (+ jaw/brow/mouth) | En az yan dönme, en güçlü face evidence |

In [ ]:
common_v3_kwargs = dict(
    layout_lora_scales={"hermione": 0.10, "daenerys": 0.30},
    identity_lora_scales={
        "hermione": {"down": 0.15, "mid": 0.30, "up": 0.45},
        "daenerys": {"down": 0.55, "mid": 1.20, "up": 1.50},
    },
    layout_ctrl_scale=1.00,
    identity_ctrl_scale=0.95,
    refine_strength=0.40,
    refine_steps=28, layout_steps=28, seed=42,
    use_regional_attention=True,
    use_spatial_lora_gate=True,
    spatial_gate_block_floor={"down": 0.15, "mid": 0.08, "up": 0.00},
    use_bg_lock=True, bg_lock_ratio=0.40, bg_lock_padding=24,
    bg_lock_in_layout=True, bg_lock_in_identity=False,
    identity_regions=identity_regions,
)

_, s2_simple = generate_two_stage(pipe, identities, pose_simple, **common_v3_kwargs)
print("\n[Stage 2 with face-aware SIMPLE pose]");  display(s2_simple.resize((448, 448)))
show_evaluator_scores("pose_simple", s2_simple)

_, s2_dense = generate_two_stage(pipe, identities, pose_dense, **common_v3_kwargs)
print("\n[Stage 2 with face-aware DENSE pose]");   display(s2_dense.resize((448, 448)))
show_evaluator_scores("pose_dense", s2_dense)

## Bölüm 6 — BG Lock Ratio Ablation

3. kişi sorunu için lock süresinin etkisi. `bg_lock_ratio` Stage 1 step sayısının kaçta kaçında background dondurulur.

| Ratio | Beklenti |
|---|---|
| 0.00 | 3. kişi serbest, baseline |
| 0.20 | hafif iyileşme |
| 0.40 | sweet spot |
| 0.60 | aşırı, foreground edges blur olabilir |

In [ ]:
bg_results = {}
for ratio in [0.00, 0.20, 0.40, 0.60]:
    print(f"\n=== bg_lock_ratio={ratio} ===")
    kw = {**common_v3_kwargs, "use_bg_lock": (ratio > 0), "bg_lock_ratio": ratio}
    _, s2 = generate_two_stage(pipe, identities, pose_dense, **kw)
    bg_results[f"bg_{ratio:.2f}"] = s2
    display(s2.resize((448, 448)))
    show_evaluator_scores(f"bg_{ratio:.2f}", s2)

## Bölüm 7 — Multi-Seed Stability (rapora final tablo)

v3 main config × 3 seed → mean ± std rapor.

In [ ]:
v3_seeds = [42, 123, 777]
v3_seed_imgs = {}

for s in v3_seeds:
    print(f"\n=== v3 seed {s} ===")
    kw = {**common_v3_kwargs, "seed": s}
    _, s2 = generate_two_stage(pipe, identities, pose_dense, **kw)
    v3_seed_imgs[s] = s2
    display(s2.resize((448, 448)))
    show_evaluator_scores(f"v3_seed{s}", s2)

## Bölüm 8 — Ablation Tablosu + Save + ZIP

In [ ]:
import pandas as pd

def collect(label, image):
    fc = face_scorer.score_image(image, identity_regions)
    pc = pose_scorer.score_image(image, target_keypoints, identity_regions)
    ac = attr_scorer.score_image(image, identity_regions)
    sim_h, sim_d = evaluator.detect_and_assign_faces(
        image, ref_embeds["hermione"], ref_embeds["daenerys"]
    )
    n_faces = detected_face_count(image)
    rows = []
    for k, eval_sim in zip(["hermione", "daenerys"], [sim_h, sim_d]):
        rows.append({
            "method":         label,
            "identity":       k,
            "arcface_self":   round(fc[k]["arcface"], 4),
            "dominance":      round(fc[k]["dominance"], 4),
            "pose":           round(pc[k]["pose"], 4),
            "attr_margin":    round(ac[k]["margin"], 4),
            "eval_arcface":   round(eval_sim, 4),
            "face_count":     n_faces,
        })
    return rows

rows = []
rows += collect("v3_main_stage1", stage1)
rows += collect("v3_main_stage2", stage2)
rows += collect("pose_simple",    s2_simple)
rows += collect("pose_dense",     s2_dense)
for label, img in bg_results.items():
    rows += collect(label, img)
for s, img in v3_seed_imgs.items():
    rows += collect(f"v3_seed{s}", img)

df = pd.DataFrame(rows)
out = "data/results/darf_v3"
os.makedirs(out, exist_ok=True)
df.to_csv(f"{out}/ablation.csv", index=False)
df

In [ ]:
# Image kaydet + ZIP
stage1.save(f"{out}/v3_main_stage1.png")
stage2.save(f"{out}/v3_main_stage2.png")
s2_simple.save(f"{out}/pose_simple.png")
s2_dense.save(f"{out}/pose_dense.png")
for label, img in bg_results.items():
    img.save(f"{out}/{label}.png")
for s, img in v3_seed_imgs.items():
    img.save(f"{out}/v3_seed{s}.png")
pose_simple.save(f"{out}/skeleton_simple.png")
pose_dense.save(f"{out}/skeleton_dense.png")

import shutil
from IPython.display import FileLink
shutil.make_archive("/workspace/darf_v3_results", "zip", out)
print("Zip ready: /workspace/darf_v3_results.zip")
FileLink("/workspace/darf_v3_results.zip")

## Akademik Çerçeve

v3 = **C-style two-stage with face-aware pose conditioning + background-locked layout stage**.

v1 v2'nin failure mode'larına nokta atışı:

| Sorun | v3 mekanizması |
|---|---|
| Daenerys yan profile bias | `pose_dense` — explicit jaw/brow/mouth/eye lineları + nose-neck centerline → ControlNet face orientation'ı net algılar |
| Üçüncü kişi hallüsinasyonu | Stage 1 BG lock (ratio=0.40) + Stage 1 minimum LoRA (`hermione=0.10, daenerys=0.30`) → orijinal noise canvas pose dışında dondurulur |
| Hermione dominance / Daenerys saç leak | Stage 2 spatial gate per-block floor (`down=0.15, mid=0.08, up=0.00`) + asimetrik LoRA (Daenerys mid=1.2, up=1.5) → kompozisyon coherence + identity isolation |
| Stage 2 pose drift | `identity_ctrl_scale=0.95` + `refine_strength=0.40` → Stage 1 layout %60 korunur |

Final iddiamız:
> *"DARF v3 decomposes multi-LoRA failure into pose-orientation, spatial routing, and background freedom problems, addressing each with a dedicated mechanism. We show on three seeds that this composition stabilises identity, attribute, and people-count metrics simultaneously, where v1 and v2 trade them off."*